In [3]:
import os
import mlflow
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

%matplotlib inline
%config InlineBackend.figure_format = 'retina' #or 'retina'

In [4]:
# Set paths
fasta_folder = os.path.join(os.getcwd(), os.pardir, 'data', 'raw-data', 'fasta')
interim_folder = os.path.join(os.getcwd(), os.pardir, 'data', 'interim')
models_folder = os.path.join(os.getcwd(), os.pardir, 'models')
processed_folder = os.path.join(os.getcwd(), os.pardir, 'data', 'processed')
img_folder = os.path.join(os.getcwd(), os.pardir, 'reports', 'figures')

In [5]:
# Define organisms dataset to be used
dataset_names = (
        'Arabidopsis_non_tata',
        'Arabidopsis_tata',
        'Bacillus',
        'Ecoli',
        'Human_non_tata',
        'Mouse_non_tata',
        'Mouse_tata',
    )

In [ ]:
# Open MLflow experiment and get runs data for each dataset and model type
uri = 'file:' + os.path.join(os.getcwd(), 'mlruns')
mlflow.set_tracking_uri(uri)
experiment_ids = dict() # One experiment per dataset
for dataset_name in dataset_names:
    experiment = mlflow.get_experiment_by_name(dataset_name)
    experiment_ids[dataset_name] = experiment.experiment_id

In [ ]:
# Get runs data for each dataset
runs_data = {}
for dataset_name in dataset_names:
    runs_data[dataset_name] = mlflow.search_runs(
        experiment_ids=experiment_ids[dataset_name],
        # filter_string="tags.mlflow.runName LIKE '%test%'"
    )

In [ ]:
cols = ['tags.mlflow.runName',
        'metrics.MCC-avg', 'metrics.MCC-max', 'metrics.MCC-min',
        'tags.Prop Name', 'tags.Model Name', 'tags.Model ID']
for dataset_name, run in runs_data.items():
    print(dataset_name)
    df = run[cols]
    print(df.sort_values(by='metrics.MCC-avg', ascending=False).head(5))
    break

In [ ]:
# Get runs data for each dataset and model type
runs_data = {}
for dataset_name in dataset_names:
    runs_data[dataset_name] = {}
    for model_type in ('cnn', 'lstm', 'mlp'):
        runs_data[dataset_name][model_type] = mlflow.search_runs(
            experiment_ids=experiment_id,
            filter_string=f"tags.dataset = '{dataset_name}' and tags.model_type = '{model_type}'"
        )

In [ ]:
# Get best runs for each dataset and model type
best_runs = {}
for dataset_name in dataset_names:
    best_runs[dataset_name] = {}
    for model_type in ('cnn', 'lstm', 'mlp'):
        best_runs[dataset_name][model_type] = runs_data[dataset_name][model_type].sort_values(by='metrics.test_accuracy', ascending=False).iloc[0]